# NSGA-II Multi-objective Optimization

Uses Pymoo NSGA-II optimization to maximize conversion/gasoline yield and minimize coke formation using the ANN surrogate model.


In [ ]:
!pip -q install pymoo

import numpy as np
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.optimize import minimize

def surrogate_predict(x_row):
    # x_row: [SiO2/Al2O3, ZSM5_wt%, Temp_C, CTO]
    x = np.array(x_row, dtype=np.float32).reshape(1, -1)
    x_s = sx.transform(x)
    with torch.no_grad():
        y_s = model(torch.tensor(x_s)).numpy()
    y = sy.inverse_transform(y_s)[0]
    # outputs: [Conversion, Gasoline, Coke, DryGas, BottomFrac]
    return y

class FCCOpt(Problem):
    def __init__(self):
        super().__init__(
            n_var=4, n_obj=4, n_constr=0,
            xl=np.array([23, 13, 450, 4]),
            xu=np.array([80, 30, 550, 6])
        )
    def _evaluate(self, X, out, *args, **kwargs):
        F = []
        for row in X:
            conv, gaso, coke, drygas, bottom = surrogate_predict(row)
            # minimize objectives (so negate gasoline for maximize)
            F.append([
                -gaso,
                coke,
                drygas,
                bottom
            ])
        out["F"] = np.array(F)

problem = FCCOpt()
algo = NSGA2(pop_size=80, sampling=FloatRandomSampling())

res = minimize(problem, algo, ('n_gen', 120), verbose=False)
pareto = res.X

print("Top 5 Pareto candidates:")
for i in range(5):
    print(pareto[i], "=>", surrogate_predict(pareto[i]))


In [ ]:
import pandas as pd

# --- Table 6 (design variables) ---
design = pd.DataFrame({
    "Run": list(range(1, 16)),
    "SiO2_Al2O3": [50,80,80,80,50,80,23,50,23,50,23,23,50,50,50],
    "ZSM5_wt_pct": [22,30,22,13,30,22,22,22,13,30,30,22,13,13,22],
    "Temp_C": [500,500,450,500,450,550,550,500,500,550,500,450,550,450,500],
    "CTO": [5,6,5,4,6,5,5,5,4,6,6,5,4,4,5],
})

# --- Table 12 (measured responses) ---
resp = pd.DataFrame([
    [1,  73.73, 51.64, 4.14, 15.05, 23.28, 7.27, 14.82, 9.17],
    [2,  85.58, 64.78, 4.74, 19.21, 29.31, 8.34, 12.47, 11.52],
    [3,  71.53, 53.85, 2.42, 14.99, 25.30, 10.02, 7.66, 11.14],
    [4,  74.71, 56.30, 3.17, 15.20, 27.93, 10.63, 7.78, 10.00],
    [5,  59.08, 46.67, 2.93, 13.79, 21.73, 6.65, 5.77, 8.22],
    [6,  66.22, 53.87, 5.27, 18.00, 21.99, 6.33, 6.02, 8.61],
    [7,  70.84, 57.06, 6.57, 17.23, 23.70, 5.90, 7.87, 9.56],
    [8,  66.57, 40.77, 2.37, 12.05, 18.88, 7.63, 18.17, 7.47],
    [9,  63.59, 53.22, 5.67, 12.52, 23.47, 6.14, 4.23, 11.55],
    [10, 72.95, 52.09, 6.24, 12.48, 22.88, 6.66, 14.20, 10.50],
    [11, 72.02, 43.86, 5.47, 10.15, 19.04, 8.01, 20.14, 9.21],
    [12, 65.47, 43.52, 1.45, 11.33, 23.35, 10.27, 11.67, 7.39],
    [13, 73.08, 49.91, 5.36, 14.05, 21.83, 7.73, 15.44, 8.68],
    [14, 81.27, 50.21, 2.42, 12.84, 24.88, 12.54, 18.52, 10.08],
    [15, 70.50, 42.90, 2.77, 10.74, 21.80, 9.06, 18.54, 7.58],
], columns=[
    "Run","MassBalance_pct","Conversion_pct","DryGas_pct","LPG_pct",
    "GasolineNaphtha_pct","LCO_pct","BottomFrac_pct","Coke_pct"
])

df = design.merge(resp, on="Run")
df.to_csv("fcc_biooil_zeolite_dataset.csv", index=False)
print(df.head())


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df[["SiO2_Al2O3","ZSM5_wt_pct","Temp_C","CTO"]].values.astype(np.float32)
Y = df[["Conversion_pct","GasolineNaphtha_pct","Coke_pct","DryGas_pct","BottomFrac_pct"]].values.astype(np.float32)

Xtr, Xte, Ytr, Yte = train_test_split(X, Y, test_size=0.25, random_state=7)

sx = StandardScaler().fit(Xtr)
sy = StandardScaler().fit(Ytr)

Xtr_s = sx.transform(Xtr); Xte_s = sx.transform(Xte)
Ytr_s = sy.transform(Ytr); Yte_s = sy.transform(Yte)

Xtr_t = torch.tensor(Xtr_s); Ytr_t = torch.tensor(Ytr_s)
Xte_t = torch.tensor(Xte_s); Yte_t = torch.tensor(Yte_s)

model = nn.Sequential(
    nn.Linear(4, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, Y.shape[1])
)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(1500):
    model.train()
    pred = model(Xtr_t)
    loss = loss_fn(pred, Ytr_t)
    opt.zero_grad(); loss.backward(); opt.step()

model.eval()
with torch.no_grad():
    te_pred = sy.inverse_transform(model(Xte_t).numpy())
print("Sample predictions (test):")
print(te_pred[:3])


In [ ]:
!pip -q install pymoo

import numpy as np
from pymoo.core.problem import Problem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.sampling.rnd import FloatRandomSampling
from pymoo.optimize import minimize

def surrogate_predict(x_row):
    # x_row: [SiO2/Al2O3, ZSM5_wt%, Temp_C, CTO]
    x = np.array(x_row, dtype=np.float32).reshape(1, -1)
    x_s = sx.transform(x)
    with torch.no_grad():
        y_s = model(torch.tensor(x_s)).numpy()
    y = sy.inverse_transform(y_s)[0]
    # outputs: [Conversion, Gasoline, Coke, DryGas, BottomFrac]
    return y

class FCCOpt(Problem):
    def __init__(self):
        super().__init__(
            n_var=4, n_obj=4, n_constr=0,
            xl=np.array([23, 13, 450, 4]),
            xu=np.array([80, 30, 550, 6])
        )
    def _evaluate(self, X, out, *args, **kwargs):
        F = []
        for row in X:
            conv, gaso, coke, drygas, bottom = surrogate_predict(row)
            # minimize objectives (so negate gasoline for maximize)
            F.append([
                -gaso,
                coke,
                drygas,
                bottom
            ])
        out["F"] = np.array(F)

problem = FCCOpt()
algo = NSGA2(pop_size=80, sampling=FloatRandomSampling())

res = minimize(problem, algo, ('n_gen', 120), verbose=False)
pareto = res.X

print("Top 5 Pareto candidates:")
for i in range(5):
    print(pareto[i], "=>", surrogate_predict(pareto[i]))


In [ ]:
# !pip -q install ezdxf shapely networkx torch torch-geometric pymoo pandas numpy scikit-learn

import ezdxf, numpy as np, pandas as pd, networkx as nx
from shapely.geometry import LineString, Point
from shapely.strtree import STRtree

dxf_path = "/content/CR11210004_P&ID OF IN SITU VAPOUR UPGRADATION.dxf"
doc = ezdxf.readfile(dxf_path)
msp = doc.modelspace()

# 1) Collect INSERT blocks (symbols/equipment/instruments)
insert_rows = []
for e in msp.query("INSERT"):
    p = e.dxf.insert
    insert_rows.append({
        "id": f"ins_{len(insert_rows)}",
        "block": e.dxf.name,
        "x": float(p.x), "y": float(p.y),
        "layer": e.dxf.layer
    })
ins = pd.DataFrame(insert_rows)

# 2) Collect LINE/LWPOLYLINE as pipes (geometry)
pipe_lines = []
for e in msp:
    if e.dxftype() == "LINE":
        p1, p2 = e.dxf.start, e.dxf.end
        pipe_lines.append(LineString([(p1.x,p1.y),(p2.x,p2.y)]))
    elif e.dxftype() == "LWPOLYLINE":
        pts = [(p[0], p[1]) for p in e.get_points()]
        if len(pts) >= 2:
            pipe_lines.append(LineString(pts))

# 3) Normalize + remove outliers (you already did similar)
XY = ins[["x","y"]].to_numpy()
med = np.median(XY, axis=0)
dist = np.linalg.norm(XY - med, axis=1)
keep = dist < np.quantile(dist, 0.98)    # drop far outliers
ins = ins.loc[keep].reset_index(drop=True)

XY = ins[["x","y"]].to_numpy()
mu, sig = XY.mean(axis=0), XY.std(axis=0) + 1e-9
ins["xn"] = (ins["x"]-mu[0])/sig[0]
ins["yn"] = (ins["y"]-mu[1])/sig[1]


In [ ]:
tol = 5.0  # CAD units tolerance; tune
points = [Point(xy) for xy in ins[["x","y"]].to_numpy()]
tree = STRtree(points)

G = nx.Graph()
for i, r in ins.iterrows():
    G.add_node(r["id"], block=r["block"], layer=r["layer"], x=r["xn"], y=r["yn"])

# pipe-based connections: if two inserts both lie near same pipe segment, connect
for line in pipe_lines:
    # buffer the line to catch nearby symbols
    buf = line.buffer(tol)
    idx = [j for j,p in enumerate(points) if buf.contains(p)]
    if len(idx) >= 2:
        for a in idx:
            for b in idx:
                if a < b:
                    G.add_edge(ins.loc[a,"id"], ins.loc[b,"id"], etype="pipe")


In [ ]:
reactor_tags = {
    "R-100": {"role":"acidification_1"},
    "R-200": {"role":"acidification_2"},
    "R-300": {"role":"acidification_3"},
    "R-101": {"role":"catalytic_fixed_bed"}
}

# Later: map these tags to node IDs (by nearest TEXT entity or by block name)
# For demo: you manually choose node ids once and store them.


In [ ]:
# Factors: acidity (SiO2/Al2O3), Temperature, CTO
# Responses: conversion, dry_gas, lpg, gasoline, lco, bottoms, coke, etc.

data = [
 # run, acidity, T, CTO, massbal, conv, drygas, lpg, gasoline, lco, bottoms, coke
 [1,  50, 500, 5, 73.73, 51.64, 4.14, 15.05, 23.28, 7.27, 14.82, 9.17],
 [2,  80, 500, 6, 85.58, 64.78, 4.74, 19.21, 29.31, 8.34, 12.47, 11.52],
 # ...continue filling from Table 12...
]
df = pd.DataFrame(data, columns=[
 "run","acidity","T_C","CTO","mass_balance","conversion",
 "dry_gas","LPG","gasoline_naphtha","LCO","bottoms","coke"
])


In [ ]:
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
import numpy as np

# Suppose you trained a surrogate model f(x)->y (even a RandomForest for demo)
# x = [acidity, T_C, CTO]

class FCCProblem(ElementwiseProblem):
    def __init__(self, model):
        super().__init__(n_var=3, n_obj=3, n_constr=0,
                         xl=np.array([23, 450, 4]),
                         xu=np.array([80, 550, 6]))
        self.model = model

    def _evaluate(self, x, out, *args, **kwargs):
        y = self.model.predict(x.reshape(1,-1))[0]  # returns [gasoline, coke, drygas] for demo
        gasoline, coke, drygas = y
        out["F"] = np.array([-gasoline, coke, drygas])  # maximize gasoline => minimize -gasoline

# algorithm = NSGA2(pop_size=80)
# res = minimize(FCCProblem(model), algorithm, ("n_gen", 120), seed=1, verbose=True)
# res.F gives Pareto front, res.X gives settings
